# Metric Exploration: What Predicts Nonlinear Encoding Benefit?

Goal: Find a metric that collapses different (n, m) combinations onto a single predictive curve for nonlinear gain.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from scipy import stats

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Copy core code from main notebook

class Autoencoder(nn.Module):
    def __init__(self, n: int, m: int, l: int, activation=nn.ReLU):
        super().__init__()
        self.n = n
        self.m = m
        self.l = l
        
        encoder_layers = []
        for i in range(l):
            encoder_layers.append(nn.Linear(n, n))
            encoder_layers.append(activation())
        encoder_layers.append(nn.Linear(n, m))
        self.encoder = nn.Sequential(*encoder_layers)
        
        decoder_layers = []
        decoder_layers.append(nn.Linear(m, n))
        decoder_layers.append(activation())
        for i in range(l - 1):
            decoder_layers.append(nn.Linear(n, n))
            decoder_layers.append(activation())
        if l > 0:
            decoder_layers.append(nn.Linear(n, n))
        self.decoder = nn.Sequential(*decoder_layers)
        
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


def generate_sparse_data(n_samples: int, n_features: int, sparsity: float = 0.1) -> torch.Tensor:
    mask = (torch.rand(n_samples, n_features) < sparsity).float()
    values = torch.rand(n_samples, n_features)
    return (mask * values).to(device)


def train_autoencoder(model, n_steps=5000, batch_size=256, sparsity=0.1, lr=1e-3, verbose=False):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    iterator = tqdm(range(n_steps)) if verbose else range(n_steps)
    for step in iterator:
        x = generate_sparse_data(batch_size, model.n, sparsity)
        optimizer.zero_grad()
        x_recon, z = model(x)
        loss = nn.functional.mse_loss(x_recon, x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return losses


def measure_encoding_linearity(model, n_samples=1000, sparsity=0.1):
    model.eval()
    
    with torch.no_grad():
        x = generate_sparse_data(n_samples, model.n, sparsity)
        z = model.encode(x)
        
        x_with_bias = torch.cat([x, torch.ones(n_samples, 1, device=device)], dim=1)
        W_linear = torch.linalg.lstsq(x_with_bias, z).solution
        z_linear = x_with_bias @ W_linear
        
        z_var = z.var(dim=0).sum()
        residual_var = (z - z_linear).var(dim=0).sum()
        linearity_score = 1 - (residual_var / z_var).item()
        
        x_recon_full, _ = model(x)
        x_recon_linear = model.decode(z_linear)
        
        mse_full = nn.functional.mse_loss(x_recon_full, x).item()
        mse_linear = nn.functional.mse_loss(x_recon_linear, x).item()
        
    return {
        'linearity_score': linearity_score,
        'mse_full': mse_full,
        'mse_linear': mse_linear,
        'nonlinear_gain': (mse_linear - mse_full) / (mse_linear + 1e-8)
    }


def run_experiment(n, m, l, sparsity=0.1, n_steps=5000, verbose=False):
    model = Autoencoder(n, m, l).to(device)
    losses = train_autoencoder(model, n_steps=n_steps, sparsity=sparsity, verbose=verbose)
    linearity_metrics = measure_encoding_linearity(model, sparsity=sparsity)
    
    return {
        'n': n, 'm': m, 'l': l, 'sparsity': sparsity,
        'final_loss': np.mean(losses[-100:]),
        **linearity_metrics
    }

## Run Sweep with Wider Range of n Values

We need varied n to see if metrics collapse the curves.

In [ ]:
# Sweep with more n values to test metric collapse
results = []
sparsity = 0.1

configs = []
for n in [16, 32, 64, 128, 256]:
    for m in [2, 4, 8, 16, 32, 64]:
        if m < n:  # valid compression
            for l in [2, 3]:  # fix depth to reduce noise
                configs.append((n, m, l))

print(f"Running {len(configs)} configurations...")

for n, m, l in tqdm(configs):
    res = run_experiment(n, m, l, sparsity=sparsity, n_steps=3000, verbose=False)
    results.append(res)

df = pd.DataFrame(results)
print(f"Collected {len(df)} results")
df.head()

## Define Candidate Metrics

In [ ]:
sparsity = 0.1

def add_metrics(df, sparsity):
    """Add all candidate metrics to dataframe."""
    df = df.copy()
    
    # Basic
    df['ratio'] = df['n'] / df['m']  # original
    df['log_ratio'] = np.log(df['n'] / df['m'])
    
    # JL-inspired
    df['jl_slack'] = df['m'] / np.log(df['n'])
    df['jl_excess'] = df['m'] - np.log(df['n'])  # additive version
    
    # Sparsity-aware
    df['active_features'] = sparsity * df['n']  # expected active
    df['dims_per_active'] = df['m'] / (sparsity * df['n'])
    df['active_ratio'] = (sparsity * df['n']) / df['m']  # inverse
    df['log_active_ratio'] = np.log((sparsity * df['n']) / df['m'])
    
    # Superposition-inspired (from Anthropic's work)
    df['superposition'] = (df['n'] - df['m']) / df['m']  # excess features per dim
    df['log_superposition'] = np.log1p(df['superposition'])
    
    # Capacity metrics (sphere packing intuition)
    # In R^m, can fit ~exp(c*m) nearly-orthogonal vectors
    df['exp_capacity_ratio'] = df['n'] / np.exp(0.5 * df['m'])  # features vs exponential capacity
    df['log_exp_capacity'] = np.log(df['n']) - 0.5 * df['m']
    
    # Polynomial capacity (softer than exponential)
    df['poly_capacity'] = df['n'] / (df['m'] ** 2)
    df['sqrt_capacity'] = df['n'] / (df['m'] ** 1.5)
    
    # Information-theoretic
    # Bits needed: ~log2(n choose k) where k = sparsity * n
    k = sparsity * df['n']
    # Approximate: k * log2(n/k) bits
    df['bits_needed'] = k * np.log2(df['n'] / k + 1)
    df['bits_per_dim'] = df['bits_needed'] / df['m']
    
    # Combined metrics
    df['combined_1'] = np.log(df['n']) / df['m']  # log features per dim
    df['combined_2'] = (sparsity * df['n']) / (df['m'] * np.log(df['m'] + 1))  # active per log-capacity
    
    return df

df = add_metrics(df, sparsity)
print(f"Added {len([c for c in df.columns if c not in ['n', 'm', 'l', 'sparsity', 'final_loss', 'linearity_score', 'mse_full', 'mse_linear', 'nonlinear_gain']])} metrics")

## Evaluate Metrics: Curve Collapse Test

A good metric should make points with different n values fall on the same curve.

In [ ]:
def evaluate_metric_collapse(df, metric, target='nonlinear_gain', n_bins=8):
    """
    Evaluate how well a metric collapses different n values onto one curve.
    
    Returns:
    - correlation: Pearson correlation with target
    - collapse_score: Low within-bin variance across n values (lower = better collapse)
    - residual_var: Variance not explained by the metric
    """
    # Basic correlation
    corr = df[metric].corr(df[target])
    
    # Bin the metric and check if different n values have similar target within bins
    try:
        df['_bin'] = pd.qcut(df[metric], q=n_bins, duplicates='drop')
        
        # For each bin, compute variance of target across different n values
        within_bin_var = []
        for bin_val in df['_bin'].unique():
            bin_data = df[df['_bin'] == bin_val]
            if len(bin_data['n'].unique()) > 1:
                # Variance of mean target across n values within this bin
                means_by_n = bin_data.groupby('n')[target].mean()
                within_bin_var.append(means_by_n.var())
        
        collapse_score = np.mean(within_bin_var) if within_bin_var else np.nan
    except:
        collapse_score = np.nan
    
    # Residual variance after linear fit
    slope, intercept, r, p, se = stats.linregress(df[metric], df[target])
    predicted = slope * df[metric] + intercept
    residual_var = ((df[target] - predicted) ** 2).mean()
    
    return {
        'metric': metric,
        'correlation': corr,
        'collapse_score': collapse_score,
        'residual_var': residual_var,
        'r_squared': r ** 2
    }


# Evaluate all metrics
metric_cols = [c for c in df.columns if c not in 
               ['n', 'm', 'l', 'sparsity', 'final_loss', 'linearity_score', 
                'mse_full', 'mse_linear', 'nonlinear_gain', '_bin', 'active_features']]

evaluations = []
for metric in metric_cols:
    eval_result = evaluate_metric_collapse(df, metric)
    evaluations.append(eval_result)

eval_df = pd.DataFrame(evaluations)
eval_df = eval_df.sort_values('r_squared', ascending=False)
print("Metric Evaluation (sorted by R-squared):")
print(eval_df.to_string(index=False))

In [ ]:
# Visualize top metrics
top_metrics = eval_df.head(6)['metric'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, metric in zip(axes.flat, top_metrics):
    for n in sorted(df['n'].unique()):
        subset = df[df['n'] == n]
        ax.scatter(subset[metric], subset['nonlinear_gain'], 
                   label=f'n={n}', s=50, alpha=0.7)
    
    r2 = eval_df[eval_df['metric'] == metric]['r_squared'].values[0]
    ax.set_xlabel(metric)
    ax.set_ylabel('Nonlinear Gain')
    ax.set_title(f'{metric}\nR² = {r2:.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Top 6 Metrics by R-squared', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Visual collapse test: do different n values overlay?
# Compare worst (ratio) vs best metric

best_metric = eval_df.iloc[0]['metric']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Original ratio
ax = axes[0]
for n in sorted(df['n'].unique()):
    subset = df[df['n'] == n]
    ax.scatter(subset['ratio'], subset['nonlinear_gain'], label=f'n={n}', s=60, alpha=0.7)
ax.set_xlabel('n/m ratio')
ax.set_ylabel('Nonlinear Gain')
ax.set_title('Original Ratio: Curves Separate by n')
ax.legend()

# Right: Best metric
ax = axes[1]
for n in sorted(df['n'].unique()):
    subset = df[df['n'] == n]
    ax.scatter(subset[best_metric], subset['nonlinear_gain'], label=f'n={n}', s=60, alpha=0.7)
ax.set_xlabel(best_metric)
ax.set_ylabel('Nonlinear Gain')
ax.set_title(f'Best Metric ({best_metric}): Better Collapse?')
ax.legend()

plt.tight_layout()
plt.show()

## Try Fitting Custom Functional Forms

In [ ]:
from scipy.optimize import curve_fit

# Try fitting: nonlinear_gain = f(n, m, sparsity)
# with different functional forms

def form_1(X, a, b, c):
    """n^a / m^b"""
    n, m = X
    return c * (n ** a) / (m ** b)

def form_2(X, a, b):
    """(sparsity * n - m) / m, scaled"""
    n, m = X
    return a * (0.1 * n - m) / m + b

def form_3(X, a, b, c):
    """log(n)/m form"""
    n, m = X
    return a * np.log(n) / (m ** b) + c

def form_4(X, a, b, c):
    """(sparsity*n)^a / m^b"""
    n, m = X
    return c * ((0.1 * n) ** a) / (m ** b)

def form_5(X, a, b):
    """sigmoid of log ratio"""
    n, m = X
    z = a * np.log(n / m) + b
    return 1 / (1 + np.exp(-z))

# Fit each form
X = (df['n'].values, df['m'].values)
y = df['nonlinear_gain'].values

forms = [
    ('n^a / m^b', form_1, [0.5, 1, 0.1]),
    ('(s*n - m)/m', form_2, [0.1, 0]),
    ('log(n) / m^b', form_3, [1, 1, 0]),
    ('(s*n)^a / m^b', form_4, [0.5, 1, 0.1]),
    ('sigmoid(log ratio)', form_5, [1, -1]),
]

print("Fitting functional forms:")
print("-" * 60)

best_form = None
best_r2 = -1

for name, func, p0 in forms:
    try:
        popt, _ = curve_fit(func, X, y, p0=p0, maxfev=10000)
        y_pred = func(X, *popt)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot
        
        print(f"{name:25s}: R² = {r2:.4f}, params = {[f'{p:.3f}' for p in popt]}")
        
        if r2 > best_r2:
            best_r2 = r2
            best_form = (name, func, popt)
    except Exception as e:
        print(f"{name:25s}: Failed - {e}")

print("-" * 60)
print(f"Best form: {best_form[0]} with R² = {best_r2:.4f}")

In [ ]:
# Visualize best fit
if best_form:
    name, func, popt = best_form
    df['fitted_metric'] = func((df['n'].values, df['m'].values), *popt)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Predicted vs actual
    ax = axes[0]
    ax.scatter(df['fitted_metric'], df['nonlinear_gain'], alpha=0.6)
    ax.plot([0, df['nonlinear_gain'].max()], [0, df['nonlinear_gain'].max()], 'r--', label='y=x')
    ax.set_xlabel('Predicted (fitted metric)')
    ax.set_ylabel('Actual Nonlinear Gain')
    ax.set_title(f'Best Fit: {name}\nR² = {best_r2:.4f}')
    ax.legend()
    
    # Collapse check
    ax = axes[1]
    for n in sorted(df['n'].unique()):
        subset = df[df['n'] == n]
        ax.scatter(subset['fitted_metric'], subset['nonlinear_gain'], label=f'n={n}', s=60, alpha=0.7)
    ax.set_xlabel('Fitted Metric')
    ax.set_ylabel('Nonlinear Gain')
    ax.set_title('Collapse Test with Fitted Metric')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## Summary & Recommendation

In [ ]:
print("="*60)
print("SUMMARY: Best Metrics for Predicting Nonlinear Gain")
print("="*60)

print("\nTop 5 simple metrics (by R²):")
for i, row in eval_df.head(5).iterrows():
    print(f"  {row['metric']:25s}: R² = {row['r_squared']:.4f}")

if best_form:
    print(f"\nBest fitted form: {best_form[0]}")
    print(f"  R² = {best_r2:.4f}")
    print(f"  Parameters: {[f'{p:.3f}' for p in best_form[2]]}")

print("\n" + "="*60)
print("RECOMMENDATION for main notebook:")
print("="*60)
best_simple = eval_df.iloc[0]['metric']
print(f"\nUse metric: '{best_simple}'")
print(f"Formula: [see add_metrics function above]")